# Multiregion hourly IDW interpolation

Elevation-aware IDW gridding for station predictors + MRoS categorical surfaces.
Supports CA (Sierra Nevada / Lake Tahoe) and CO (Colorado Mountains) via a single
region toggle.  Mirror of the multiregion kriging script but using IDW throughout.

**Change `REGION` in Cell 2 to switch between CA and CO.**

### What this script does
1. Interpolates continuous station predictors (`temp_air`, `temp_dew`, `temp_wet`, `rh`)
   using dynamic per-hour lapse-rate detrend/retrend IDW.
2. Interpolates categorical MRoS observations as one-hot class-support fields
   using elevation-aware 3-D IDW (`z_scale=4.0`).
3. Saves full-grid predictor surfaces as `p_snow / p_mix / p_rain`.
4. Builds a pointwise leave-one-out (LOOCV) MRoS prediction table for
   leakage-safe ML predictors at raw MRoS station locations.
5. Chunks output day-by-day and flushes to a single compressed NetCDF,
   with resume support if the run is interrupted.

### Notes
- Raw MRoS observations remain the labels for later ML.
- Gridded MRoS surfaces (`p_snow/p_mix/p_rain`) and LOOCV predictions are
  predictors only.
- LOOCV and MRoS surfaces are only computed for `mros_active_months` to
  avoid wasted computation and dilution by off-season empty hours.
- Lapse rate is estimated from `temp_air` and applied to all temperature
  variables; `rh` is interpolated without lapse correction.


In [ ]:
# ====================================================================
# REGION TOGGLE — change this to switch between CA and CO runs
# ====================================================================
REGION = "CA"   # "CA" = Sierra Nevada / Lake Tahoe
                 # "CO" = Colorado Mountains

REGION_CONFIG = {
    "CA": {
        "label":              "California: Sierra Nevada / Lake Tahoe",
        "utm_crs":            "EPSG:26911",
        "dem_file":           "CA_DEM_AOI_1km.tif",
        "station_sub":        "CA",
        "imerg_sub":          "CA",
        "mros_active_months": (10, 11, 12, 1, 2, 3, 4, 5),
        "aoi_lonlat": [
            (-119.45505750721992, 39.65343608043361),
            (-121.27878797084242, 39.66189413918429),
            (-119.11448133630248, 36.726935737063016),
            (-118.49924696303225, 37.235952484988736),
            (-119.46604383531404, 38.37304030164334),
        ],
    },
    "CO": {
        "label":              "Colorado Mountains",
        "utm_crs":            "EPSG:32613",
        "dem_file":           "CO_DEM_AOI_1km.tif",
        "station_sub":        "CO",
        "imerg_sub":          "CO",
        "mros_active_months": (9, 10, 11, 12, 1, 2, 3, 4, 5, 6),
        "aoi_lonlat": [
            (-105.19885928678391, 40.62046076499234),
            (-106.88927700287375, 40.555465783967925),
            (-107.66078646416787, 38.79540171139857),
            (-104.87856373593310, 38.77382201116306),
        ],
    },
}

rcfg = REGION_CONFIG[REGION]
print(f"Running region: {REGION} — {rcfg['label']}")

In [ ]:
# ====================================================================
# CONFIG
# ====================================================================
from __future__ import annotations

import shutil
import warnings
from itertools import groupby
from pathlib import Path
from typing import Optional, Sequence, Tuple

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio as rio
import rioxarray  # noqa: F401
import xarray as xr
from pyproj import CRS, Transformer
from rasterio.transform import rowcol as rio_rowcol
from rasterio.transform import xy as rio_xy
from rasterio.warp import (
    Resampling,
    calculate_default_transform,
    reproject,
    transform_bounds,
)
from scipy.spatial import cKDTree
from shapely.geometry import box, MultiPoint
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, f1_score, log_loss
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=RuntimeWarning)

BASE_DIR = Path().resolve().parent   # notebook lives in Scripts/
print("BASE_DIR:", BASE_DIR)


def build_config(base_dir: Path = BASE_DIR) -> dict:
    cfg = {
        # --- timing ---
        "wy_start":           "2022-10-01T00:00:00Z",
        "wy_end":             "2026-05-01T23:59:59Z",
        "mros_active_months": rcfg["mros_active_months"],

        # --- inputs ---
        "dem_path":           base_dir / f"Data/Elevation/{rcfg['dem_file']}",
        "stations_parquet":   base_dir / f"outputs/assimilated/{REGION}/hourly_data/stations_hourly.parquet",
        "mros_parquet":       base_dir / f"outputs/assimilated/{REGION}/hourly_data/mros_hourly.parquet",

        # --- outputs ---
        "out_dir":            base_dir / f"outputs/interpolated/{REGION}/IDW_refactored",

        # --- CRS ---
        "proj_fallback":      rcfg["utm_crs"],

        # --- IDW tuning ---
        "idw_power":          2.0,
        "k_nearest":          8,
        "min_points_global":  3,
        "min_points_lapse":   5,
        "lapse_degC_per_m":   -0.005,
        "lapse_bounds":       (-0.009, 0.002),
        "mros_vertical_scale": 4.0,
        "eps":                1e-6,

        # --- processing flags ---
        "save_processed_parquet": True,
        "reuse_processed":        True,
    }
    cfg["out_dir"].mkdir(parents=True, exist_ok=True)
    (cfg["out_dir"] / "processed_inputs").mkdir(parents=True, exist_ok=True)
    return cfg


# per-variable settings: which variables get lapse correction
VAR_CONFIG = {
    "temp_air": {"min_points": 4, "apply_lapse": True},
    "temp_dew": {"min_points": 4, "apply_lapse": True},
    "temp_wet": {"min_points": 4, "apply_lapse": True},
    "rh":       {"min_points": 4, "apply_lapse": False},
}

PHASE_COL_CANDIDATES = ("phase", "mros_phase", "phase_class", "ptype", "precip_phase", "phase_label")
PHASE_ALIASES = {
    "snow": "snow", "s": "snow", "sn": "snow", "solid": "snow",
    "mix": "mix", "mixed": "mix", "transition": "mix", "m": "mix",
    "rain": "rain", "r": "rain", "liquid": "rain",
    "rain_snow": "mix", "snow_rain": "mix", "rainsnow": "mix",
}
PHASES       = ("snow", "mix", "rain")
SUPPORT_COLS = ["mros_support_snow", "mros_support_mix", "mros_support_rain"]

In [ ]:
# ====================================================================
# TIME HELPERS
# ====================================================================
def to_utc(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, errors="coerce", utc=True).dt.floor("h")


def hourly_index(start_iso: str, end_iso: str) -> pd.DatetimeIndex:
    return pd.date_range(
        start=pd.to_datetime(start_iso), end=pd.to_datetime(end_iso),
        freq="h", tz="UTC",
    )

In [ ]:
# ====================================================================
# DEM / GRID UTILITIES
# ====================================================================
def load_dem(path: Path, target_crs: CRS) -> Tuple[np.ndarray, dict, CRS]:
    """Load DEM, reprojecting to target_crs if needed. Returns (dem, profile, crs)."""
    with rio.open(path) as src:
        src_crs = CRS.from_user_input(src.crs)
        tgt_crs = CRS.from_user_input(target_crs)
        if src_crs == tgt_crs:
            dem     = src.read(1).astype(np.float32)
            profile = src.profile.copy()
            profile["crs"] = tgt_crs.to_wkt()
            return dem, profile, tgt_crs
        transform, width, height = calculate_default_transform(
            src_crs, tgt_crs, src.width, src.height, *src.bounds
        )
        profile = src.profile.copy()
        profile.update({"crs": tgt_crs.to_wkt(), "transform": transform,
                        "width": width, "height": height, "dtype": "float32"})
        dem = np.full((height, width), np.nan, dtype=np.float32)
        reproject(
            source=rio.band(src, 1), destination=dem,
            src_transform=src.transform, src_crs=src_crs,
            dst_transform=transform, dst_crs=tgt_crs,
            resampling=Resampling.bilinear,
            src_nodata=src.nodata, dst_nodata=np.nan,
        )
        return dem, profile, tgt_crs


def grid_centers(profile: dict) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return (x_centers, y_centers, grid_xy) for a rasterio profile."""
    T  = profile["transform"]
    xs = T.c + (np.arange(profile["width"])  + 0.5) * T.a
    ys = T.f + (np.arange(profile["height"]) + 0.5) * T.e
    X, Y = np.meshgrid(xs, ys)
    return xs, ys, np.column_stack([X.ravel(), Y.ravel()])


def sample_dem_at_xy(
    xs: np.ndarray, ys: np.ndarray,
    dem_data: np.ndarray, dem_profile: dict,
) -> np.ndarray:
    rows, cols = rio_rowcol(dem_profile["transform"], xs, ys)
    rows = np.asarray(rows)
    cols = np.asarray(cols)
    valid = (
        (rows >= 0) & (rows < dem_data.shape[0]) &
        (cols >= 0) & (cols < dem_data.shape[1])
    )
    out = np.full(len(xs), np.nan, dtype=np.float32)
    out[valid] = dem_data[rows[valid], cols[valid]]
    nodata = dem_profile.get("nodata")
    if nodata is not None:
        out[np.isclose(out, nodata)] = np.nan
    return out


def aoi_poly_from_dem(dem_path: Path) -> object:
    """Derive AOI polygon in WGS84 from DEM bounds."""
    with rio.open(dem_path) as src:
        b = src.bounds
        wgs = transform_bounds(
            src.crs, "EPSG:4326",
            b.left, b.bottom, b.right, b.top,
            densify_pts=21,
        )
    return box(wgs[0], wgs[1], wgs[2], wgs[3])

In [ ]:
# ====================================================================
# PROJECTION / ELEVATION HELPERS
# ====================================================================
def project_lonlat(
    df: pd.DataFrame, proj_crs: CRS,
) -> Tuple[np.ndarray, np.ndarray]:
    tfm = Transformer.from_crs("EPSG:4326", proj_crs, always_xy=True)
    x, y = tfm.transform(df["lon"].values, df["lat"].values)
    return np.asarray(x, dtype=float), np.asarray(y, dtype=float)


def attach_projected_xy_and_dem(
    df: pd.DataFrame, proj_crs: CRS,
    dem_data: np.ndarray, dem_profile: dict,
) -> pd.DataFrame:
    out    = df.copy()
    x, y   = project_lonlat(out, proj_crs)
    out["x"] = x
    out["y"] = y
    if "elev" not in out.columns:
        out["elev"] = np.nan
    need = out["elev"].isna()
    if need.any():
        out.loc[need, "elev"] = sample_dem_at_xy(
            out.loc[need, "x"].to_numpy(),
            out.loc[need, "y"].to_numpy(),
            dem_data, dem_profile,
        )
    return out


def filter_points_to_aoi(df: pd.DataFrame, aoi_poly) -> pd.DataFrame:
    g    = gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df["lon"], df["lat"]),
        crs="EPSG:4326",
    )
    mask = g.intersects(
        gpd.GeoSeries([aoi_poly], crs="EPSG:4326").iloc[0]
    )
    return df.loc[mask.values].copy()

In [ ]:
# ====================================================================
# MRoS PREPROCESSING
# ====================================================================
def infer_phase_column(df: pd.DataFrame) -> Optional[str]:
    for col in PHASE_COL_CANDIDATES:
        if col in df.columns:
            return col
    return None


def canonicalize_phase_value(v) -> Optional[str]:
    if pd.isna(v):
        return None
    s = str(v).strip().lower()
    return PHASE_ALIASES.get(s, s if s in PHASES else None)


def convert_legacy_proxy_to_phase(v) -> Optional[str]:
    if pd.isna(v):
        return None
    try:
        fv = float(v)
    except Exception:
        return None
    if np.isclose(fv,   0.0): return "snow"
    if np.isclose(fv,  50.0): return "mix"
    if np.isclose(fv, 100.0): return "rain"
    return None


def prepare_mros_onehot(
    mros_df: pd.DataFrame,
    proj_crs: CRS,
    dem_data: np.ndarray,
    dem_profile: dict,
) -> pd.DataFrame:
    df = mros_df.copy()
    phase_col = infer_phase_column(df)
    if phase_col is not None:
        df["mros_phase"] = df[phase_col].map(canonicalize_phase_value)
    elif "mros_plp_proxy" in df.columns:
        warnings.warn("No categorical phase column found; falling back to mros_plp_proxy.")
        df["mros_phase"] = df["mros_plp_proxy"].map(convert_legacy_proxy_to_phase)
    else:
        raise ValueError(
            "No categorical MRoS phase column found and no legacy mros_plp_proxy.\n"
            f"Columns present: {list(df.columns)}"
        )
    df = attach_projected_xy_and_dem(df, proj_crs, dem_data, dem_profile)
    df["mros_support_snow"] = (df["mros_phase"] == "snow").astype(float)
    df["mros_support_mix"]  = (df["mros_phase"] == "mix" ).astype(float)
    df["mros_support_rain"] = (df["mros_phase"] == "rain").astype(float)
    return df


def dedupe_station_hourly(
    df: pd.DataFrame, value_cols: Sequence[str],
) -> pd.DataFrame:
    grp = ["hour_utc", "lon", "lat"]
    def reducer(g: pd.DataFrame) -> pd.Series:
        out = {"hour_utc": g.name[0], "lon": g.name[1], "lat": g.name[2]}
        for c in value_cols:
            vals = pd.to_numeric(g[c], errors="coerce").dropna() if c in g.columns else pd.Series(dtype=float)
            out[c] = float(vals.mean()) if len(vals) else np.nan
        return pd.Series(out)
    return df.groupby(grp, dropna=False, sort=False).apply(reducer).reset_index(drop=True)


def dedupe_mros_hourly(df: pd.DataFrame) -> pd.DataFrame:
    grp = ["hour_utc", "lon", "lat"]
    def reducer(g: pd.DataFrame) -> pd.Series:
        phases = g["mros_phase"].dropna().tolist()
        out    = {"hour_utc": g.name[0], "lon": g.name[1], "lat": g.name[2]}
        if not phases:
            out["mros_phase"]        = None
            out["is_soft_duplicate"] = False
            for c in SUPPORT_COLS:
                out[c] = np.nan
            return pd.Series(out)
        counts = pd.Series(phases).value_counts(normalize=True)
        out["mros_phase"]        = counts.idxmax()
        out["is_soft_duplicate"] = len(counts) > 1
        out["mros_support_snow"] = float(counts.get("snow", 0.0))
        out["mros_support_mix"]  = float(counts.get("mix",  0.0))
        out["mros_support_rain"] = float(counts.get("rain", 0.0))
        return pd.Series(out)
    return df.groupby(grp, dropna=False, sort=False).apply(reducer).reset_index(drop=True)

In [ ]:
# ====================================================================
# LAPSE RATE HELPERS
# ====================================================================
def estimate_lapse_rate(
    st_df: pd.DataFrame,
    temp_col: str  = "temp_air",
    elev_col: str  = "elev",
    default_lapse: float = -0.005,
    min_points: int = 5,
    bounds: tuple   = (-0.009, 0.002),
) -> float:
    """Per-hour lapse rate from linear regression of temp vs elevation."""
    use = st_df.dropna(subset=[temp_col, elev_col])
    if len(use) < min_points:
        return default_lapse
    try:
        model = LinearRegression().fit(
            use[[elev_col]].values.astype(float),
            use[temp_col].values.astype(float),
        )
        slope = float(model.coef_[0])
        return slope if bounds[0] <= slope <= bounds[1] else default_lapse
    except Exception:
        return default_lapse

In [ ]:
# ====================================================================
# IDW INTERPOLATION HELPERS
# ====================================================================
def _normalize_weights(d: np.ndarray, power: float) -> np.ndarray:
    """Compute IDW weights from distances; handles exact-hit (zero distance)."""
    with np.errstate(divide="ignore"):
        w = 1.0 / np.power(d, power)
    w[np.isinf(w)] = 1e12
    w[~np.isfinite(w)] = 0.0
    w_sum = w.sum(axis=1, keepdims=True)
    return np.divide(w, w_sum, out=np.zeros_like(w), where=w_sum > 0)


def idw_detrend_by_lapse(
    hour_points: pd.DataFrame,
    grid_xy: np.ndarray,
    grid_elev: np.ndarray,
    proj_crs: CRS,
    value_col: str,
    station_elev_col: str  = "elev",
    lapse_degC_per_m: float = -0.005,
    idw_power: float = 2.0,
    k: int  = 8,
    min_points: int = 3,
) -> np.ndarray:
    """
    Lapse-rate detrend at station elevations → IDW of residuals →
    retrend with lapse at grid elevations.  Pass lapse=0 for RH.
    """
    pts = hour_points.dropna(subset=[value_col, "lon", "lat", station_elev_col]).copy()
    if pts.empty or pts[value_col].notna().sum() < min_points:
        return np.full(grid_elev.shape, np.nan, dtype=np.float32)
    px, py  = project_lonlat(pts, proj_crs)
    P       = np.column_stack([px, py])
    vj      = pts[value_col].to_numpy(dtype=float)
    zj      = pts[station_elev_col].to_numpy(dtype=float)
    v_norm  = vj - lapse_degC_per_m * zj
    tree    = cKDTree(P)
    dists, idxs = tree.query(grid_xy, k=min(k, len(P)))
    if dists.ndim == 1:
        dists, idxs = dists[:, None], idxs[:, None]
    w_norm  = _normalize_weights(dists, idw_power)
    valid   = np.sum(w_norm > 0, axis=1)
    result  = np.sum(w_norm * v_norm[idxs], axis=1)
    result[valid < min_points] = np.nan
    return (result + lapse_degC_per_m * grid_elev).astype(np.float32)


def idw_3d_support(
    hour_points: pd.DataFrame,
    grid_xy: np.ndarray,
    grid_elev: np.ndarray,
    proj_crs: CRS,
    value_col: str,
    station_elev_col: str = "elev",
    idw_power: float = 2.0,
    k: int  = 8,
    min_points: int = 2,
    z_scale: float  = 4.0,
) -> np.ndarray:
    """
    Elevation-aware 3-D IDW for MRoS support fields.
    z_scale stretches the vertical axis so 1 m elevation ≈ z_scale m horizontal.
    """
    pts = hour_points.dropna(subset=[value_col, "lon", "lat", station_elev_col]).copy()
    if pts.empty or pts[value_col].notna().sum() < min_points:
        return np.full(grid_elev.shape, np.nan, dtype=np.float32)
    px, py  = project_lonlat(pts, proj_crs)
    Pxy     = np.column_stack([px, py])
    Pz      = pts[station_elev_col].to_numpy(dtype=float)
    v       = pts[value_col].to_numpy(dtype=float)
    tree    = cKDTree(Pxy)
    d_xy, idxs = tree.query(grid_xy, k=min(k, len(Pxy)))
    if d_xy.ndim == 1:
        d_xy, idxs = d_xy[:, None], idxs[:, None]
    dz      = grid_elev[:, None] - Pz[idxs]
    d_eff   = np.sqrt(np.square(d_xy) + np.square(z_scale * dz))
    w_norm  = _normalize_weights(d_eff, idw_power)
    valid   = np.sum(np.isfinite(d_eff), axis=1)
    result  = np.sum(w_norm * v[idxs], axis=1)
    result[valid < min_points] = np.nan
    return result.astype(np.float32)


def normalize_support_cube(
    snow: np.ndarray, mix: np.ndarray, rain: np.ndarray, eps: float,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Clip to [0,1] and normalize each pixel to sum to 1."""
    S     = np.stack([np.clip(snow, 0., 1.), np.clip(mix, 0., 1.), np.clip(rain, 0., 1.)], axis=0)
    total = np.nansum(S, axis=0)
    valid = total > eps
    out   = np.full_like(S, np.nan, dtype=np.float32)
    out[:, valid] = (S[:, valid] / total[valid]).astype(np.float32)
    return out[0], out[1], out[2]


def interpolate_support_vector_at_point(
    train_df: pd.DataFrame,
    target_lon: float,
    target_lat: float,
    target_elev: float,
    proj_crs: CRS,
    idw_power: float = 2.0,
    k: int  = 8,
    min_points: int = 2,
    z_scale: float  = 4.0,
    eps: float      = 1e-6,
) -> dict:
    """Single-point 3-D IDW of support fields for LOOCV."""
    pts = train_df.dropna(subset=["lon", "lat", "elev"] + SUPPORT_COLS).copy()
    if len(pts) < min_points:
        return {c: np.nan for c in SUPPORT_COLS}
    px, py = project_lonlat(pts, proj_crs)
    tx, ty = Transformer.from_crs("EPSG:4326", proj_crs, always_xy=True).transform(
        [target_lon], [target_lat]
    )
    Pxy  = np.column_stack([px, py])
    Pz   = pts["elev"].to_numpy(dtype=float)
    tree = cKDTree(Pxy)
    d_xy, idxs = tree.query([[tx[0], ty[0]]], k=min(k, len(Pxy)))
    d_xy = d_xy.ravel()
    idxs = idxs.ravel()
    dz   = target_elev - Pz[idxs]
    d_eff = np.sqrt(d_xy**2 + (z_scale * dz)**2)
    with np.errstate(divide="ignore"):
        w = 1.0 / np.power(d_eff, idw_power)
    w[np.isinf(w)] = 1e12
    w_sum = w.sum()
    if w_sum <= eps:
        return {c: np.nan for c in SUPPORT_COLS}
    w /= w_sum
    result = {}
    for c in SUPPORT_COLS:
        v = pts[c].to_numpy(dtype=float)
        result[c] = float(np.sum(w * v[idxs]))
    # normalize triplet
    total = sum(max(result[c], 0.) for c in SUPPORT_COLS)
    if total > eps:
        result = {c: max(result[c], 0.) / total for c in SUPPORT_COLS}
    else:
        result = {c: np.nan for c in SUPPORT_COLS}
    return result

In [ ]:
# ====================================================================
# LOOCV
# ====================================================================
def loocv_mros_hour(
    mros_t: pd.DataFrame,
    proj_crs: CRS,
    idw_power: float,
    k: int,
    min_points: int,
    z_scale: float,
    eps: float,
) -> Tuple[pd.DataFrame, dict]:
    """Leave-one-out cross-validation of MRoS support interpolation for one hour."""
    cols_needed = ["lon", "lat", "elev", "mros_phase"] + SUPPORT_COLS
    df = mros_t[
        mros_t["mros_phase"].isin(PHASES)
    ].dropna(subset=cols_needed).copy()

    if len(df) < max(min_points + 1, 3):
        return pd.DataFrame(), {
            "n_points": len(df), "n_eval": 0,
            "accuracy": np.nan, "macro_f1": np.nan, "multiclass_logloss": np.nan,
        }

    rows = []
    for i in range(len(df)):
        test  = df.iloc[i]
        train = df.drop(df.index[i])
        pred  = interpolate_support_vector_at_point(
            train_df=train,
            target_lon=float(test["lon"]),
            target_lat=float(test["lat"]),
            target_elev=float(test["elev"]),
            proj_crs=proj_crs,
            idw_power=idw_power,
            k=k,
            min_points=min_points,
            z_scale=z_scale,
            eps=eps,
        )
        if any(pd.isna(list(pred.values()))):
            continue
        probs      = [pred["mros_support_snow"], pred["mros_support_mix"], pred["mros_support_rain"]]
        pred_phase = PHASES[int(np.argmax(probs))]
        rows.append({
            "lon":               float(test["lon"]),
            "lat":               float(test["lat"]),
            "elev":              float(test["elev"]),
            "obs_phase":         test["mros_phase"],
            "pred_phase":        pred_phase,
            "mros_p_snow_loocv": pred["mros_support_snow"],
            "mros_p_mix_loocv":  pred["mros_support_mix"],
            "mros_p_rain_loocv": pred["mros_support_rain"],
            "obs_p_snow":        float(test["mros_support_snow"]),
            "obs_p_mix":         float(test["mros_support_mix"]),
            "obs_p_rain":        float(test["mros_support_rain"]),
            "pred_max_prob":     float(np.max(probs)),
            "pred_entropy":      float(
                -np.sum(np.clip(probs, eps, 1.) * np.log(np.clip(probs, eps, 1.)))
            ),
            "pred_correct":      int(pred_phase == test["mros_phase"]),
        })

    details = pd.DataFrame(rows)
    if details.empty:
        return details, {
            "n_points": len(df), "n_eval": 0,
            "accuracy": np.nan, "macro_f1": np.nan, "multiclass_logloss": np.nan,
        }

    y_true  = details["obs_phase"].values
    y_pred  = details["pred_phase"].values
    y_prob  = details[["mros_p_snow_loocv", "mros_p_mix_loocv", "mros_p_rain_loocv"]].values
    y_ohe   = details[["obs_p_snow", "obs_p_mix", "obs_p_rain"]].values
    metrics = {
        "n_points": len(df),
        "n_eval":   len(details),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=list(PHASES), average="macro"),
    }
    try:
        metrics["multiclass_logloss"] = log_loss(y_ohe, y_prob)
    except Exception:
        metrics["multiclass_logloss"] = np.nan
    return details, metrics


def summarize_loocv(points_df: pd.DataFrame) -> pd.DataFrame:
    if points_df.empty:
        return pd.DataFrame([{
            "scope": "overall", "n_eval": 0,
            "accuracy": np.nan, "macro_f1": np.nan, "multiclass_logloss": np.nan,
        }])
    overall = {
        "scope":   "overall",
        "n_eval":  len(points_df),
        "accuracy": accuracy_score(points_df["obs_phase"], points_df["pred_phase"]),
        "macro_f1": f1_score(
            points_df["obs_phase"], points_df["pred_phase"],
            labels=list(PHASES), average="macro",
        ),
    }
    try:
        overall["multiclass_logloss"] = log_loss(
            points_df[["obs_p_snow", "obs_p_mix", "obs_p_rain"]].values,
            points_df[["mros_p_snow_loocv", "mros_p_mix_loocv", "mros_p_rain_loocv"]].values,
        )
    except Exception:
        overall["multiclass_logloss"] = np.nan

    by_hour = []
    for hour, g in points_df.dropna(subset=["hour_utc"]).groupby("hour_utc"):
        row = {"scope": "hourly", "hour_utc": hour, "n_eval": len(g)}
        row["accuracy"] = accuracy_score(g["obs_phase"], g["pred_phase"]) if len(g) else np.nan
        row["macro_f1"] = f1_score(
            g["obs_phase"], g["pred_phase"], labels=list(PHASES), average="macro",
        ) if len(g) else np.nan
        try:
            row["multiclass_logloss"] = log_loss(
                g[["obs_p_snow", "obs_p_mix", "obs_p_rain"]].values,
                g[["mros_p_snow_loocv", "mros_p_mix_loocv", "mros_p_rain_loocv"]].values,
            )
        except Exception:
            row["multiclass_logloss"] = np.nan
        by_hour.append(row)
    return pd.concat(
        [pd.DataFrame([overall]), pd.DataFrame(by_hour)], ignore_index=True
    )

In [ ]:
# ====================================================================
# DATA PREPARATION
# ====================================================================
def prepare_inputs(cfg: dict, force_rebuild: bool = False):
    proc_dir  = cfg["out_dir"] / "processed_inputs"
    st_proc   = proc_dir / "stations_processed.parquet"
    mros_proc = proc_dir / "mros_processed.parquet"
    proj_crs  = CRS.from_user_input(cfg["proj_fallback"])
    dem_data, dem_profile, proj_crs = load_dem(cfg["dem_path"], proj_crs)

    if cfg["reuse_processed"] and not force_rebuild and st_proc.exists() and mros_proc.exists():
        print("Reusing cached processed parquets.")
        return pd.read_parquet(st_proc), pd.read_parquet(mros_proc), dem_data, dem_profile, proj_crs

    st   = pd.read_parquet(cfg["stations_parquet"])
    mros = pd.read_parquet(cfg["mros_parquet"])
    print(f"Stations columns : {list(st.columns)}")
    print(f"MRoS columns     : {list(mros.columns)}")

    st["hour_utc"]   = to_utc(st["hour_utc"])
    mros["hour_utc"] = to_utc(mros["hour_utc"])

    time_mask_st = (
        (st["hour_utc"]   >= pd.to_datetime(cfg["wy_start"])) &
        (st["hour_utc"]   <= pd.to_datetime(cfg["wy_end"]))
    )
    time_mask_m = (
        (mros["hour_utc"] >= pd.to_datetime(cfg["wy_start"])) &
        (mros["hour_utc"] <= pd.to_datetime(cfg["wy_end"]))
    )
    st   = st.loc[time_mask_st].copy()
    mros = mros.loc[time_mask_m].copy()

    # spatial filter to DEM AOI
    aoi_poly = aoi_poly_from_dem(cfg["dem_path"])
    st   = filter_points_to_aoi(st,   aoi_poly)
    mros = filter_points_to_aoi(mros, aoi_poly)

    available_vars = [c for c in ["temp_air", "temp_dew", "temp_wet", "rh"] if c in st.columns]
    if not available_vars:
        raise ValueError("No temperature/RH variables found in station parquet.")

    st = st[["hour_utc", "lon", "lat"] + available_vars].copy()
    st = dedupe_station_hourly(st, available_vars)
    st = attach_projected_xy_and_dem(st, proj_crs, dem_data, dem_profile)

    mros = prepare_mros_onehot(mros, proj_crs, dem_data, dem_profile)
    keep_cols = ["hour_utc", "lon", "lat", "elev", "mros_phase"] + SUPPORT_COLS
    mros = mros[[c for c in keep_cols if c in mros.columns]].copy()
    mros = dedupe_mros_hourly(mros)
    mros = attach_projected_xy_and_dem(mros, proj_crs, dem_data, dem_profile)

    st   = st.dropna(subset=["x", "y"])
    mros = mros.dropna(subset=["x", "y"])

    if cfg["save_processed_parquet"]:
        st.to_parquet(st_proc,   index=False)
        mros.to_parquet(mros_proc, index=False)
        print(f"Saved processed parquets to {proc_dir}")

    return st, mros, dem_data, dem_profile, proj_crs

In [ ]:
# ====================================================================
# MAIN INTERPOLATION LOOP  (chunked, resumable)
# ====================================================================
def interpolate_all_hours(
    st_hr, mros_hr, dem_data, dem_profile, proj_crs, cfg,
):
    import netCDF4 as nc4

    x_centers, y_centers, grid_xy = grid_centers(dem_profile)
    # ensure y_centers are monotonically increasing (south→north)
    if y_centers[0] > y_centers[-1]:
        y_centers = y_centers[::-1]
        dem_data  = dem_data[::-1, :]
        X, Y      = np.meshgrid(x_centers, y_centers)
        grid_xy   = np.column_stack([X.ravel(), Y.ravel()])

    H, W          = dem_data.shape
    valid_points  = np.isfinite(dem_data).ravel()
    grid_xy_valid = grid_xy[valid_points]
    grid_elev     = dem_data.ravel().astype(np.float32)
    grid_elev_valid = grid_elev[valid_points]

    times    = hourly_index(cfg["wy_start"], cfg["wy_end"])
    out_vars = ["temp_air", "temp_dew", "temp_wet", "rh",
                "p_snow",   "p_mix",    "p_rain"]
    # only keep vars present in the station data
    station_vars = [v for v in ["temp_air", "temp_dew", "temp_wet", "rh"]
                    if v in st_hr.columns]

    ckpt_dir  = cfg["out_dir"] / "hourly_chunks"
    loocv_dir = cfg["out_dir"] / "loocv_chunks"
    ckpt_dir.mkdir(exist_ok=True)
    loocv_dir.mkdir(exist_ok=True)

    final_nc = cfg["out_dir"] / "hourly_predictors_1km_IDW.nc"
    BATCH    = 30

    # ------------------------------------------------------------------
    # Guard: validate / delete a stale final_nc
    # ------------------------------------------------------------------
    if final_nc.exists():
        try:
            ds_check = xr.open_dataset(final_nc)
            existing_start = pd.Timestamp(ds_check.time.values[0])
            existing_end   = pd.Timestamp(ds_check.time.values[-1])
            ds_check.close()
            expected_start = pd.Timestamp(cfg["wy_start"])
            expected_end   = pd.Timestamp(cfg["wy_end"])
            if existing_start != expected_start or existing_end.date() != expected_end.date():
                print(
                    f"  WARNING: final_nc time range {existing_start.date()} → {existing_end.date()} "
                    f"doesn't match config — deleting and rebuilding"
                )
                final_nc.unlink()
            else:
                print(f"  Existing final_nc looks valid: {final_nc}")
        except Exception:
            print(f"  WARNING: {final_nc} appears corrupted — deleting and rebuilding")
            final_nc.unlink()

    days_written = len(list(ckpt_dir.glob("*.nc")))
    if days_written > 0:
        print(f"  Resuming with {days_written} pending unassembled chunk(s)")

    # ------------------------------------------------------------------
    # Flush helper — assembles pending day-chunks into final_nc
    # ------------------------------------------------------------------
    def flush_chunks():
        pending = sorted(ckpt_dir.glob("*.nc"))
        if not pending:
            return
        try:
            if not final_nc.exists():
                opened   = [xr.open_dataset(f) for f in pending]
                ds_batch = xr.concat(opened, dim="time").sortby("time")
                encoding = {v: {"zlib": True, "complevel": 4} for v in out_vars if v in ds_batch}
                encoding["time"] = {"units": "hours since 2022-10-01", "calendar": "standard"}
                ds_batch.to_netcdf(final_nc, unlimited_dims=["time"], encoding=encoding)
                ds_batch.close()
                for _ds in opened:
                    _ds.close()
                # write CF spatial_ref variable
                proj_crs_obj = CRS.from_user_input(rcfg["utm_crs"])
                with nc4.Dataset(final_nc, "a") as ds_nc:
                    crs_var = ds_nc.createVariable("spatial_ref", "i4")
                    crs_var.crs_wkt           = proj_crs_obj.to_wkt()
                    crs_var.grid_mapping_name = proj_crs_obj.to_cf()["grid_mapping_name"]
                    crs_var.spatial_ref       = proj_crs_obj.to_wkt()
                    for v in out_vars:
                        if v in ds_nc.variables:
                            ds_nc.variables[v].grid_mapping = "spatial_ref"
                    ds_nc.crs            = rcfg["utm_crs"]
                    ds_nc.region         = REGION
                    ds_nc.region_label   = rcfg["label"]
                    ds_nc.description    = "Hourly station interpolation (lapse-rate IDW) + MRoS 3-D IDW"
                    ds_nc.mros_note      = "p_snow/p_mix/p_rain are predictor surfaces only."
                    ds_nc.idw_power      = cfg["idw_power"]
                    ds_nc.k_nearest      = cfg["k_nearest"]
                    ds_nc.mros_z_scale   = cfg["mros_vertical_scale"]
            else:
                origin = pd.Timestamp("2022-10-01")
                with nc4.Dataset(final_nc, "a") as dst:
                    current_len = len(dst.variables["time"])
                    for f in pending:
                        chunk_ds = xr.open_dataset(f)
                        times_in_chunk = chunk_ds.time.values
                        chunk_ds.close()
                        hours_since = np.array([
                            (pd.Timestamp(t) - origin).total_seconds() / 3600
                            for t in times_in_chunk
                        ], dtype=np.float64)
                        n = len(hours_since)
                        with nc4.Dataset(f, "r") as src:
                            dst.variables["time"][current_len:current_len + n] = hours_since
                            for var in out_vars:
                                if var in src.variables:
                                    dst.variables[var][current_len:current_len + n] = src.variables[var][:]
                        current_len += n
            for f in pending:
                f.unlink()
            print(f"  Flushed {len(pending)} chunk(s) → {final_nc.name}")
        except Exception as e:
            print(f"  WARNING: flush failed ({e}) — chunks retained")

    # ------------------------------------------------------------------
    # Step 1 — interpolate day-by-day
    # ------------------------------------------------------------------
    days = [
        (date, list(hour_group))
        for date, hour_group in groupby(times, key=lambda t: pd.Timestamp(t).date())
    ]

    for date, hour_group in tqdm(days, desc=f"Interpolating {REGION} days"):
        stamp    = date.strftime("%Y%m%d")
        nc_path  = ckpt_dir  / f"{stamp}.nc"
        csv_path = loocv_dir / f"{stamp}.csv"

        if csv_path.exists() and not nc_path.exists():
            continue  # already flushed into final_nc
        if nc_path.exists() and csv_path.exists():
            days_written += 1
            if days_written % BATCH == 0:
                flush_chunks()
            continue

        day_slices, day_loocv, day_times = [], [], []

        for t in hour_group:
            st_t   = st_hr[st_hr["hour_utc"] == t].copy()
            mros_t = mros_hr[mros_hr["hour_utc"] == t].copy()
            t_month = pd.Timestamp(t).month

            # -- per-hour lapse rate (estimated from temp_air) --
            lapse_now = estimate_lapse_rate(
                st_t,
                temp_col="temp_air",
                elev_col="elev",
                default_lapse=cfg["lapse_degC_per_m"],
                min_points=cfg["min_points_lapse"],
                bounds=cfg["lapse_bounds"],
            )

            slice_vars = {v: np.full((H, W), np.nan, dtype=np.float32) for v in out_vars}

            # -- temperature and RH surfaces --
            for name in station_vars:
                vcfg   = VAR_CONFIG.get(name, {"min_points": cfg["min_points_global"], "apply_lapse": False})
                pts    = st_t[["lon", "lat", "elev", name]].dropna(subset=[name]).copy()
                if pts[name].notna().sum() < vcfg["min_points"]:
                    continue
                lapse_apply = lapse_now if vcfg.get("apply_lapse", False) else 0.0
                vals = idw_detrend_by_lapse(
                    hour_points=pts,
                    grid_xy=grid_xy_valid,
                    grid_elev=grid_elev_valid,
                    proj_crs=proj_crs,
                    value_col=name,
                    station_elev_col="elev",
                    lapse_degC_per_m=lapse_apply,
                    idw_power=cfg["idw_power"],
                    k=cfg["k_nearest"],
                    min_points=vcfg["min_points"],
                )
                # temperature sanity clamp
                if name != "rh":
                    vals = np.clip(vals, -60., 60.)
                full = np.full(H * W, np.nan, dtype=np.float32)
                full[valid_points] = vals
                slice_vars[name] = full.reshape(H, W)

            # -- MRoS surfaces + LOOCV (active season only) --
            if t_month in cfg["mros_active_months"]:
                mros_valid = mros_t.dropna(subset=["lon", "lat", "elev", "mros_phase"])
                if len(mros_valid) >= 2:
                    mros_surfaces = {}
                    for support_col in SUPPORT_COLS:
                        vals = idw_3d_support(
                            hour_points=mros_valid[["lon", "lat", "elev", support_col]].dropna(subset=[support_col]),
                            grid_xy=grid_xy_valid,
                            grid_elev=grid_elev_valid,
                            proj_crs=proj_crs,
                            value_col=support_col,
                            station_elev_col="elev",
                            idw_power=cfg["idw_power"],
                            k=cfg["k_nearest"],
                            min_points=2,
                            z_scale=cfg["mros_vertical_scale"],
                        )
                        mros_surfaces[support_col] = vals

                    snow_n, mix_n, rain_n = normalize_support_cube(
                        mros_surfaces["mros_support_snow"],
                        mros_surfaces["mros_support_mix"],
                        mros_surfaces["mros_support_rain"],
                        eps=cfg["eps"],
                    )
                    for key, arr in [("p_snow", snow_n), ("p_mix", mix_n), ("p_rain", rain_n)]:
                        full = np.full(H * W, np.nan, dtype=np.float32)
                        full[valid_points] = arr
                        slice_vars[key] = full.reshape(H, W)

                    # LOOCV
                    cv_detail, _ = loocv_mros_hour(
                        mros_t=mros_t,
                        proj_crs=proj_crs,
                        idw_power=cfg["idw_power"],
                        k=cfg["k_nearest"],
                        min_points=2,
                        z_scale=cfg["mros_vertical_scale"],
                        eps=cfg["eps"],
                    )
                    if len(cv_detail):
                        cv_detail = cv_detail.copy()
                        cv_detail["hour_utc"] = t
                        day_loocv.append(cv_detail)

            day_slices.append(slice_vars)
            day_times.append(pd.Timestamp(t).tz_localize(None))

        # write day chunk
        nc_tmp = nc_path.with_suffix(".tmp")
        xr.Dataset(
            {k: (("time", "y", "x"), np.stack([s[k] for s in day_slices]))
             for k in out_vars if k in day_slices[0]},
            coords={"time": day_times, "y": y_centers, "x": x_centers},
        ).to_netcdf(nc_tmp, encoding={k: {"zlib": True, "complevel": 4} for k in out_vars})

        loocv_day = pd.concat(day_loocv, ignore_index=True) if day_loocv else pd.DataFrame()
        loocv_day.to_csv(csv_path, index=False)
        nc_tmp.rename(nc_path)

        days_written += 1
        if days_written % BATCH == 0:
            flush_chunks()

    # flush any remaining chunks
    flush_chunks()

    # ------------------------------------------------------------------
    # Step 2 — verify final_nc
    # ------------------------------------------------------------------
    try:
        ds_check = xr.open_dataset(final_nc)
        n_times  = len(ds_check.time)
        ds_check.close()
        print(f"  final_nc verified: {n_times} timesteps")
    except Exception as e:
        raise RuntimeError(
            f"final_nc failed verification: {e}\n"
            f"LOOCV chunks retained at {loocv_dir} — do not delete manually."
        )

    # ------------------------------------------------------------------
    # Step 3 — assemble LOOCV parquet and clean up
    # ------------------------------------------------------------------
    def _safe_read_loocv(p):
        try:
            if p.stat().st_size == 0:
                return None
            df = pd.read_csv(p)
            if df.empty or "hour_utc" not in df.columns:
                return None
            df["hour_utc"] = pd.to_datetime(df["hour_utc"], utc=True, errors="coerce")
            return df
        except Exception:
            return None

    loocv_parts = [
        r for r in [_safe_read_loocv(p) for p in sorted(loocv_dir.glob("*.csv"))]
        if r is not None
    ]
    loocv_points  = pd.concat(loocv_parts, ignore_index=True) if loocv_parts else pd.DataFrame()
    if not loocv_points.empty and "hour_utc" in loocv_points.columns:
        loocv_points["hour_utc"] = pd.to_datetime(loocv_points["hour_utc"], utc=True, errors="coerce")
    loocv_summary = summarize_loocv(loocv_points)

    out_loocv_parquet = cfg["out_dir"] / "mros_loocv_point_predictions_IDW.parquet"
    out_loocv_csv     = cfg["out_dir"] / "mros_loocv_point_predictions_IDW.csv"
    out_summary_csv   = cfg["out_dir"] / "mros_loocv_summary_IDW.csv"

    if not loocv_points.empty:
        loocv_points.to_parquet(out_loocv_parquet, index=False)
        loocv_points.to_csv(out_loocv_csv,         index=False)
    loocv_summary.to_csv(out_summary_csv, index=False)

    shutil.rmtree(loocv_dir)
    remaining = list(ckpt_dir.glob("*.nc"))
    if remaining:
        print(f"  WARNING: {len(remaining)} chunk(s) still in {ckpt_dir}")
    else:
        ckpt_dir.rmdir()

    print("\nWrote:")
    print(f"  {final_nc}")
    print(f"  {out_loocv_parquet}")
    print(f"  {out_summary_csv}")

    return xr.open_dataset(final_nc), loocv_summary, loocv_points

In [ ]:
# ====================================================================
# RUN  (change REGION at the top of Cell 2 to switch CA / CO)
# ====================================================================
cfg = build_config()
print("out_dir :", cfg["out_dir"])
print("dem_path:", cfg["dem_path"])

st_hr, mros_hr, dem_data, dem_profile, proj_crs = prepare_inputs(cfg, force_rebuild=False)

print(f"\nStation rows : {len(st_hr):,}")
print(f"MRoS rows    : {len(mros_hr):,}")
print(f"Station vars : {[c for c in st_hr.columns if c not in ('hour_utc','lon','lat','x','y','elev')]}")
print(f"DEM shape    : {dem_data.shape}  (H x W)")
print(f"Grid cells   : {dem_data.size:,}  valid={int(np.isfinite(dem_data).sum()):,}")

ds, loocv_summary, loocv_points = interpolate_all_hours(
    st_hr, mros_hr, dem_data, dem_profile, proj_crs, cfg,
)

print("\nLOOCV summary:")
print(loocv_summary[loocv_summary["scope"] == "overall"].to_string(index=False))

In [ ]:
# ====================================================================
# QUICKLOOK — single timestep map
# ====================================================================
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

TARGET_DATE = "2023-03-15"   # change to any date in the run period

# reload from cache if pipeline already ran
cfg = build_config()
ds  = xr.open_dataset(cfg["out_dir"] / "hourly_predictors_1km_IDW.nc")
st_hr, mros_hr, dem_data, dem_profile, proj_crs = prepare_inputs(cfg, force_rebuild=False)

times = pd.to_datetime(ds.time.values).floor("h")
mask  = times.normalize() == pd.to_datetime(TARGET_DATE)
if not mask.any():
    raise ValueError(f"No timestep for {TARGET_DATE}")
ti     = np.where(mask)[0][0]
t_plot = times[ti]
print("Plotting:", t_plot)

xmin, xmax = ds.x.min().item(), ds.x.max().item()
ymin, ymax = ds.y.min().item(), ds.y.max().item()
extent     = [xmin, xmax, ymin, ymax]

# resolve CRS for station projection
if "spatial_ref" in ds and "crs_wkt" in ds["spatial_ref"].attrs:
    ds_crs = ds["spatial_ref"].attrs["crs_wkt"]
else:
    ds_crs = ds.attrs.get("crs", rcfg["utm_crs"])
tf = Transformer.from_crs("EPSG:4326", CRS.from_user_input(ds_crs), always_xy=True)

# project observations
def project_obs(df, t_utc):
    t_match = df["hour_utc"].dt.tz_convert(None) if df["hour_utc"].dt.tz is not None else df["hour_utc"]
    sub = df[t_match.dt.floor("h") == t_utc]
    if not len(sub): return np.array([]), np.array([])
    ox, oy = tf.transform(sub["lon"].values, sub["lat"].values)
    ox, oy = np.asarray(ox), np.asarray(oy)
    mask   = (ox >= xmin) & (ox <= xmax) & (oy >= ymin) & (oy <= ymax)
    return ox[mask], oy[mask]

st_x,  st_y  = project_obs(st_hr,   t_plot)
mo_x,  mo_y  = project_obs(mros_hr, t_plot)
print(f"Stations in grid: {len(st_x)}   MRoS in grid: {len(mo_x)}")

p_snow = ds["p_snow"].isel(time=ti).values
p_mix  = ds["p_mix" ].isel(time=ti).values
p_rain = ds["p_rain"].isel(time=ti).values
phase_pred = np.argmax(np.stack([p_snow, p_mix, p_rain]), axis=0).astype(float)
phase_pred[~np.all(np.isfinite(np.stack([p_snow, p_mix, p_rain])), axis=0)] = np.nan

vars_to_plot = [
    ("phase_pred",   phase_pred),
    ("p_snow",       p_snow),
    ("p_mix",        p_mix),
    ("p_rain",       p_rain),
    ("temp_air",     ds["temp_air"].isel(time=ti).values),
    ("temp_dew",     ds["temp_dew"].isel(time=ti).values),
    ("temp_wet",     ds["temp_wet"].isel(time=ti).values),
    ("rh",           ds["rh"].isel(time=ti).values),
]

phase_cmap = ListedColormap(["blue", "purple", "green"])
fig, axes  = plt.subplots(2, 4, figsize=(16, 8.5))
axes       = axes.flatten()
fig.suptitle(f"IDW hourly surfaces — {REGION} — {t_plot:%Y-%m-%d %H:%M UTC}", fontsize=13)

for i, (name, arr) in enumerate(vars_to_plot):
    ax = axes[i]
    kw = dict(origin="lower", extent=extent, aspect="equal")
    if name == "phase_pred":
        im = ax.imshow(np.ma.masked_invalid(arr), cmap=phase_cmap, vmin=0, vmax=2, **kw)
    elif name in ("p_snow", "p_mix", "p_rain"):
        im = ax.imshow(arr, vmin=0, vmax=1, **kw)
    elif name == "rh":
        im = ax.imshow(arr, vmin=0, vmax=100, **kw)
    else:
        im = ax.imshow(arr, **kw)
    ax.set_title(name)
    ax.set_xlabel("Easting (km)")
    ax.set_ylabel("Northing (km)")
    ax.ticklabel_format(style="plain")
    ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
    ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])
    if len(st_x):
        ax.scatter(st_x, st_y, s=12, c="white", edgecolor="black", linewidth=0.4, label="Stations")
    if len(mo_x):
        ax.scatter(mo_x, mo_y, s=25, c="red", marker="^", edgecolor="black", linewidth=0.4, label="MRoS")
    ax.legend(loc="upper right", fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

plt.tight_layout()
plt.show()

print("\nQC summary:")
for v in ["temp_air", "temp_dew", "temp_wet", "rh", "p_snow", "p_mix", "p_rain"]:
    if v not in ds: continue
    arr = ds[v].isel(time=ti).values
    fin = np.isfinite(arr)
    print(f"{v:10s} | finite={fin.mean()*100:6.2f}%  min={np.nanmin(arr):8.3f}  max={np.nanmax(arr):8.3f}  mean={np.nanmean(arr):8.3f}")
p_sum = p_snow + p_mix + p_rain
print(f"\np_snow+p_mix+p_rain: min={np.nanmin(p_sum):.4f}  max={np.nanmax(p_sum):.4f}  mean={np.nanmean(p_sum):.4f}")